# Курсовой проект по дисциплине «Технологии обработки больших данных»

**Тема:** «Исследование предпочтений в музыке на основе потоковых и текстовых данных»  
**Этап:** 2.3. Подготовка, очистка и слияние разнородных данных

На этом этапе выполняется реальная подготовка данных из источников, выбранных в п. 2.2: Spotify Top 200, MusicBrainz XML, Last.fm JSON API и The Guardian. В соответствии с методическими указаниями проводится первичный анализ, обработка пропусков и дубликатов, определение аналитических шкал и ролей признаков, аннотирование и векторизация текста, нормализация общих ключей и слияние данных.

Для API используется ограниченная выборка наиболее популярных объектов. Это позволяет воспроизводимо выполнить этап и не создавать избыточную нагрузку на открытые сервисы.

## 1. Подключение библиотек и параметры

Для основного CSV ключи не требуются. MusicBrainz также работает без API-ключа. Для Last.fm и The Guardian нужны бесплатные ключи. Они не записываются в публичный репозиторий.

Если ключи уже заданы как переменные окружения LASTFM_API_KEY и GUARDIAN_API_KEY, ноутбук использует их автоматически. Если ключа нет, соответствующий API-блок будет пропущен, а остальные части ноутбука продолжат работать.

In [ ]:
import os
import re
import time
import html
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import xml.etree.ElementTree as ET

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

SPOTIFY_URL = (
    "https://raw.githubusercontent.com/younver/"
    "spotify-top-200-dataset/main/spotify-top-200-dataset.csv"
)

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY", "")
GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY", "")

MB_ARTIST_LIMIT = 10
LASTFM_TRACK_LIMIT = 30
GUARDIAN_ARTIST_LIMIT = 8
GUARDIAN_ARTICLES_PER_ARTIST = 5

print("Last.fm key:", "задан" if LASTFM_API_KEY else "не задан")
print("Guardian key:", "задан" if GUARDIAN_API_KEY else "не задан")


### Как временно задать ключи в Colab

Перед повторным запуском API-ячеек можно выполнить отдельную временную ячейку:

    import os
    os.environ["LASTFM_API_KEY"] = "ВАШ_КЛЮЧ"
    os.environ["GUARDIAN_API_KEY"] = "ВАШ_КЛЮЧ"

Эту ячейку с реальными значениями ключей не нужно сохранять в публичный ноутбук.

## 2. Загрузка основного реляционного набора Spotify Top 200

Основной набор содержит недельные наблюдения глобального Spotify Top-200 за 2017–2021 годы. В исходном репозитории заявлено 74 661 строка и 40 столбцов. Сначала проверим фактический размер, типы данных, первые строки и пропуски.

In [ ]:
spotify_raw = pd.read_csv(SPOTIFY_URL)

print("Размер исходного набора:", spotify_raw.shape)
print("\nПервые 5 строк:")
display(spotify_raw.head())

print("\nТипы данных:")
display(
    spotify_raw.dtypes.rename("dtype")
    .astype(str)
    .to_frame()
)

print("\nПропуски:")
missing_spotify = (
    spotify_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing")
    .to_frame()
)
missing_spotify["share_%"] = (
    missing_spotify["missing"] / len(spotify_raw) * 100
).round(2)
display(missing_spotify.head(20))

print("Полных дубликатов:", spotify_raw.duplicated().sum())


## 3. Физический смысл и аналитическая роль используемых признаков

Для проверки гипотезы используются не все 40 полей, а признаки, описывающие популярность музыкального контента и возможные факторы влияния.

| Признак | Физический смысл | Аналитическая роль |
|---|---|---|
| track_id | идентификатор трека Spotify | ID / ключ |
| track_name | название композиции | категория / ключ слияния |
| artist_name | исполнитель | категория / ключ слияния |
| artist_genres | жанры исполнителя | фактор |
| album_type | тип релиза | фактор |
| explicit | наличие explicit-контента | фактор |
| collab | наличие нескольких исполнителей | фактор |
| week | неделя нахождения в чарте | время |
| release_date | дата релиза | время / фактор |
| streams | число недельных прослушиваний | основной показатель популярности |
| rank | место в чарте | показатель популярности |
| track_popularity | индекс популярности трека Spotify | показатель популярности |
| artist_popularity | индекс популярности исполнителя | фактор |
| artist_followers | число подписчиков исполнителя | фактор |
| danceability | танцевальность | фактор |
| energy | энергичность | фактор |
| valence | позитивность звучания | фактор |
| tempo | темп композиции | фактор |
| duration | длительность | фактор |
| listeners | число слушателей Last.fm | внешний показатель интереса |
| playcount | число проигрываний Last.fm | внешний показатель интереса |
| country | страна исполнителя MusicBrainz | фактор |
| type | тип исполнителя MusicBrainz | фактор |
| bodyText | текст статьи Guardian | неструктурированный контент |
| article_count | количество статей | внешний информационный фактор |

## 4. Аналитические шкалы

По типологии, используемой в курсовом проекте, признаки распределены по восьми видам шкал.

| Тип шкалы | Примеры в проекте | Обоснование |
|---|---|---|
| Номинальная (ID) | track_id, artist_id, album_id, mbid | идентификаторы, над которыми не выполняются арифметические операции |
| Контентная | bodyText, исходный XML и JSON | слабоструктурированный или неструктурированный контент |
| Категориальная | artist_name, artist_genres, album_type, country | конечное множество категорий без порядка |
| Дихотомическая | explicit, collab, mode | два состояния: да/нет |
| Порядковая | rank | значения упорядочены, но разность между соседними рангами не является равным интервалом |
| Интервальная | week, release_date, webPublicationDate, track_popularity | разности значений интерпретируемы, нулевая точка условна |
| Отношений | duration, tempo | естественный ноль, отношения значений имеют смысл |
| Абсолютная | streams, listeners, playcount, artist_followers, word_count, article_count | результат непосредственного подсчета с фиксированной единицей |

## 5. Очистка Spotify

В исходном наборе коллаборации могут быть представлены несколькими строками: по одной на каждого исполнителя. Поле pivot отмечает дополнительные строки. Для анализа на уровне «трек–неделя» оставляется основная строка pivot = 0. Также приводятся типы, обрабатываются пропуски и создаются нормализованные ключи для дальнейшего слияния.

In [ ]:
def normalize_key(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9а-яё]+", " ", value, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", value).strip()

def to_boolean(series):
    mapping = {
        True: True, False: False,
        1: True, 0: False,
        "1": True, "0": False,
        "true": True, "false": False,
        "True": True, "False": False
    }
    return series.map(mapping).astype("boolean")

spotify = spotify_raw.copy()

# Даты
for col in ["week", "release_date"]:
    if col in spotify.columns:
        spotify[col] = pd.to_datetime(spotify[col], errors="coerce")

# Числовые признаки
numeric_cols = [
    "track_popularity", "album_popularity", "artist_followers",
    "artist_popularity", "rank", "streams", "danceability",
    "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration"
]

for col in numeric_cols:
    if col in spotify.columns:
        spotify[col] = pd.to_numeric(spotify[col], errors="coerce")

# Логические признаки
for col in ["explicit", "collab", "mode", "pivot"]:
    if col in spotify.columns:
        spotify[col] = to_boolean(spotify[col])

# Оставляем основную строку трека при разбиении коллабораций
if "pivot" in spotify.columns:
    spotify = spotify[(spotify["pivot"].isna()) | (~spotify["pivot"])].copy()

# Удаляем строки без критически важных полей
required = [c for c in ["track_name", "artist_name", "week", "streams"] if c in spotify.columns]
spotify = spotify.dropna(subset=required)

# Удаляем повтор трек-неделя после фильтра pivot
dedup_keys = [c for c in ["track_id", "week"] if c in spotify.columns]
if dedup_keys:
    spotify = spotify.drop_duplicates(subset=dedup_keys, keep="first")

# Пропуски категорий
for col in ["artist_genres", "album_type", "album_label"]:
    if col in spotify.columns:
        spotify[col] = spotify[col].fillna("unknown")

# Медианная импутация аудиопризнаков
audio_cols = [
    c for c in [
        "danceability", "energy", "loudness", "speechiness",
        "acousticness", "instrumentalness", "liveness",
        "valence", "tempo", "duration"
    ]
    if c in spotify.columns
]

for col in audio_cols:
    spotify[col] = spotify[col].fillna(spotify[col].median())

# Нормализованные ключи
spotify["artist_key"] = spotify["artist_name"].map(normalize_key)
spotify["track_key"] = spotify["track_name"].map(normalize_key)

print("Размер после очистки:", spotify.shape)
print("Удалено строк:", len(spotify_raw) - len(spotify))
print("Полных дубликатов после очистки:", spotify.duplicated().sum())

display(spotify.head())


### Результат очистки Spotify

После подготовки одна строка соответствует наблюдению композиции за определенную неделю. Нормализованные artist_key и track_key будут использоваться только как технические ключи слияния; исходные названия сохраняются.

## 6. Выбор объектов для API-источников

Для открытых API используется не весь список объектов, а наиболее значимые по суммарному числу стримов. Это сокращает число запросов, соблюдает ограничения внешних сервисов и при этом сохраняет объекты, наиболее важные для исследования популярности.

In [ ]:
top_artists = (
    spotify.groupby(["artist_key", "artist_name"], as_index=False)["streams"]
    .sum()
    .sort_values("streams", ascending=False)
)

top_tracks = (
    spotify.groupby(
        ["track_key", "track_name", "artist_key", "artist_name"],
        as_index=False
    )["streams"]
    .sum()
    .sort_values("streams", ascending=False)
)

print("Топ исполнителей:")
display(top_artists.head(10))

print("Топ треков:")
display(top_tracks.head(10))


## 7. MusicBrainz: получение и разбор XML

MusicBrainz используется как иерархический XML-источник. Для небольшой выборки топ-исполнителей выполняется поиск наиболее подходящего объекта. Из XML извлекаются идентификатор MusicBrainz, имя, тип исполнителя, страна, пол, период активности и число альтернативных имен.

Для соблюдения рекомендаций MusicBrainz между запросами добавлена пауза.

In [ ]:
MB_NS = {"m": "http://musicbrainz.org/ns/mmd-2.0#"}
MB_HEADERS = {
    "User-Agent": "TOBD-course-project/1.0 (educational research)"
}

def musicbrainz_artist(name):
    response = requests.get(
        "https://musicbrainz.org/ws/2/artist/",
        params={"query": f'artist:"{name}"', "limit": 1},
        headers=MB_HEADERS,
        timeout=30
    )
    response.raise_for_status()
    root = ET.fromstring(response.text)
    artist = root.find(".//m:artist", MB_NS)

    if artist is None:
        return None

    life_span = artist.find("m:life-span", MB_NS)
    aliases = artist.findall("m:alias-list/m:alias", MB_NS)

    return {
        "artist_key": normalize_key(name),
        "mbid": artist.attrib.get("id", ""),
        "mb_type": artist.attrib.get("type", ""),
        "mb_name": artist.findtext("m:name", default="", namespaces=MB_NS),
        "country": artist.findtext("m:country", default="", namespaces=MB_NS),
        "gender": artist.findtext("m:gender", default="", namespaces=MB_NS),
        "active_begin": (
            life_span.findtext("m:begin", default="", namespaces=MB_NS)
            if life_span is not None else ""
        ),
        "active_end": (
            life_span.findtext("m:end", default="", namespaces=MB_NS)
            if life_span is not None else ""
        ),
        "alias_count": len(aliases)
    }

mb_rows = []

for artist_name in top_artists["artist_name"].head(MB_ARTIST_LIMIT):
    try:
        row = musicbrainz_artist(artist_name)
        if row:
            mb_rows.append(row)
    except Exception as exc:
        print("MusicBrainz:", artist_name, "->", exc)
    time.sleep(1.05)

musicbrainz_df = pd.DataFrame(mb_rows)

print("Получено объектов MusicBrainz:", len(musicbrainz_df))
display(musicbrainz_df.head(10))


## 8. Last.fm: подготовка JSON-данных

Для треков из Spotify запрашивается метод track.getInfo. Он позволяет получить дополнительные числовые показатели listeners и playcount, длительность, альбом и пользовательские теги. Слияние с Spotify выполняется по нормализованной паре «исполнитель + трек».

Если API-ключ не задан, создается пустая таблица нужной структуры, чтобы остальные ячейки ноутбука продолжали выполняться.

In [ ]:
LASTFM_COLUMNS = [
    "artist_key", "track_key", "lfm_mbid", "lfm_duration",
    "listeners", "playcount", "lfm_album", "lfm_tags"
]

def lastfm_track_info(artist, track):
    response = requests.get(
        "https://ws.audioscrobbler.com/2.0/",
        params={
            "method": "track.getInfo",
            "api_key": LASTFM_API_KEY,
            "artist": artist,
            "track": track,
            "autocorrect": 1,
            "format": "json"
        },
        headers={"User-Agent": "TOBD-course-project/1.0"},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()

    if "track" not in data:
        return None

    t = data["track"]
    tags = [
        x.get("name", "")
        for x in t.get("toptags", {}).get("tag", [])
    ]

    return {
        "artist_key": normalize_key(t.get("artist", {}).get("name", artist)),
        "track_key": normalize_key(t.get("name", track)),
        "lfm_mbid": t.get("mbid", ""),
        "lfm_duration": pd.to_numeric(t.get("duration"), errors="coerce"),
        "listeners": pd.to_numeric(t.get("listeners"), errors="coerce"),
        "playcount": pd.to_numeric(t.get("playcount"), errors="coerce"),
        "lfm_album": t.get("album", {}).get("title", ""),
        "lfm_tags": ", ".join(tags)
    }

lastfm_rows = []

if LASTFM_API_KEY:
    sample_tracks = top_tracks.head(LASTFM_TRACK_LIMIT)

    for _, item in sample_tracks.iterrows():
        try:
            row = lastfm_track_info(item["artist_name"], item["track_name"])
            if row:
                lastfm_rows.append(row)
        except Exception as exc:
            print("Last.fm:", item["artist_name"], "-", item["track_name"], "->", exc)
        time.sleep(0.15)

    lastfm_df = pd.DataFrame(lastfm_rows, columns=LASTFM_COLUMNS)
else:
    print("LASTFM_API_KEY не задан: запросы Last.fm пропущены.")
    lastfm_df = pd.DataFrame(columns=LASTFM_COLUMNS)

print("Получено объектов Last.fm:", len(lastfm_df))
display(lastfm_df.head())


## 9. The Guardian: сбор и аннотирование неструктурированного текста

The Guardian используется как источник неструктурированных текстов. Статьи запрашиваются по именам наиболее популярных исполнителей в разделе music.

Аннотирование выполняется путем создания дополнительных признаков:
- длина текста;
- количество слов;
- количество тегов;
- число упоминаний исполнителя;
- наличие имени исполнителя в заголовке.

После этого очищенный текст будет векторизован методом TF-IDF.

In [ ]:
GUARDIAN_COLUMNS = [
    "artist_key", "artist_name", "article_id", "title", "publication_date",
    "author", "body_text", "word_count_api", "tag_count",
    "text_length", "word_count", "artist_mention_count",
    "artist_in_title"
]

def clean_article_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^A-Za-zА-Яа-яЁё0-9'\-\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

def guardian_articles(artist):
    response = requests.get(
        "https://content.guardianapis.com/search",
        params={
            "q": artist,
            "section": "music",
            "from-date": "2017-01-01",
            "to-date": "2025-12-31",
            "page-size": GUARDIAN_ARTICLES_PER_ARTIST,
            "order-by": "relevance",
            "show-fields": "headline,byline,bodyText,wordcount",
            "show-tags": "keyword",
            "api-key": GUARDIAN_API_KEY
        },
        timeout=30
    )
    response.raise_for_status()
    return response.json()["response"]["results"]

guardian_rows = []

if GUARDIAN_API_KEY:
    for artist_name in top_artists["artist_name"].head(GUARDIAN_ARTIST_LIMIT):
        try:
            results = guardian_articles(artist_name)

            for item in results:
                fields = item.get("fields", {})
                tags = item.get("tags", [])
                title = item.get("webTitle", "")
                body = fields.get("bodyText", "")
                clean = clean_article_text(body)
                artist_norm = artist_name.lower()

                guardian_rows.append({
                    "artist_key": normalize_key(artist_name),
                    "artist_name": artist_name,
                    "article_id": item.get("id", ""),
                    "title": title,
                    "publication_date": pd.to_datetime(
                        item.get("webPublicationDate"),
                        errors="coerce"
                    ),
                    "author": fields.get("byline", ""),
                    "body_text": clean,
                    "word_count_api": pd.to_numeric(
                        fields.get("wordcount"),
                        errors="coerce"
                    ),
                    "tag_count": len(tags),
                    "text_length": len(clean),
                    "word_count": len(clean.split()),
                    "artist_mention_count": clean.count(artist_norm),
                    "artist_in_title": artist_norm in title.lower()
                })

        except Exception as exc:
            print("Guardian:", artist_name, "->", exc)

    guardian_df = pd.DataFrame(guardian_rows, columns=GUARDIAN_COLUMNS)
else:
    print("GUARDIAN_API_KEY не задан: запросы The Guardian пропущены.")
    guardian_df = pd.DataFrame(columns=GUARDIAN_COLUMNS)

print("Получено статей:", len(guardian_df))
display(guardian_df.head())


## 10. Векторизация текстов TF-IDF

Для преобразования текста в числовое представление используется TF-IDF. Это позволяет представить неструктурированные статьи в виде матрицы признаков и в дальнейшем использовать их в алгоритмах анализа данных.

Матрица хранится отдельно в разреженном формате, так как большинство значений TF-IDF равны нулю.

In [ ]:
tfidf_matrix = sparse.csr_matrix((0, 0))
tfidf_feature_names = []

if not guardian_df.empty and guardian_df["body_text"].str.len().gt(0).any():
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=100,
        min_df=1,
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform(guardian_df["body_text"])
    tfidf_feature_names = vectorizer.get_feature_names_out().tolist()

    print("Размер TF-IDF матрицы:", tfidf_matrix.shape)

    mean_weights = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
    top_idx = mean_weights.argsort()[::-1][:15]

    tfidf_top = pd.DataFrame({
        "term": np.array(tfidf_feature_names)[top_idx],
        "mean_tfidf": mean_weights[top_idx]
    })
    display(tfidf_top)
else:
    print("Тексты Guardian пока отсутствуют — TF-IDF будет построен после задания API-ключа.")


## 11. Агрегация текстовых признаков до уровня исполнителя

Spotify содержит много недельных наблюдений, а статьи Guardian являются отдельными документами. Для корректного слияния текстовые признаки агрегируются до уровня исполнителя.

In [ ]:
if not guardian_df.empty:
    guardian_artist = (
        guardian_df.groupby("artist_key", as_index=False)
        .agg(
            article_count=("article_id", "nunique"),
            avg_article_words=("word_count", "mean"),
            total_artist_mentions=("artist_mention_count", "sum"),
            avg_text_length=("text_length", "mean"),
            avg_tag_count=("tag_count", "mean")
        )
    )
else:
    guardian_artist = pd.DataFrame(columns=[
        "artist_key", "article_count", "avg_article_words",
        "total_artist_mentions", "avg_text_length", "avg_tag_count"
    ])

display(guardian_artist.head())


## 12. Общие атрибуты для слияния

Источники имеют разные форматы, поэтому перед объединением определены общие агрегаты и технические ключи.

| Источники | Общие атрибуты |
|---|---|
| Spotify + Last.fm | artist_key + track_key |
| Spotify + MusicBrainz | artist_key, дополнительно mbid после сопоставления |
| Spotify + Guardian | artist_key после агрегации статей |
| Временной анализ | week, release_date, publication_date |

artist_key и track_key создаются путем приведения строк к нижнему регистру, удаления лишних символов и повторных пробелов. Это снижает количество несовпадений из-за различий в написании.

## 13. Слияние разнородных данных

Основой итогового набора остается очищенный Spotify. Дополнительные данные подключаются левым соединением, чтобы не терять основные недельные наблюдения. После каждого слияния рассчитывается покрытие — доля строк, для которых удалось получить внешний признак.

In [ ]:
merged = spotify.copy()

# Last.fm: уровень трека
if not lastfm_df.empty:
    lastfm_unique = lastfm_df.drop_duplicates(
        subset=["artist_key", "track_key"],
        keep="last"
    )

    merged = merged.merge(
        lastfm_unique,
        on=["artist_key", "track_key"],
        how="left",
        validate="m:1"
    )
else:
    for col in [
        "lfm_mbid", "lfm_duration", "listeners",
        "playcount", "lfm_album", "lfm_tags"
    ]:
        merged[col] = np.nan

# MusicBrainz: уровень исполнителя
if not musicbrainz_df.empty:
    mb_unique = musicbrainz_df.drop_duplicates("artist_key", keep="first")
    merged = merged.merge(
        mb_unique,
        on="artist_key",
        how="left",
        validate="m:1"
    )
else:
    for col in [
        "mbid", "mb_type", "mb_name", "country",
        "gender", "active_begin", "active_end", "alias_count"
    ]:
        merged[col] = np.nan

# Guardian: уровень исполнителя
if not guardian_artist.empty:
    merged = merged.merge(
        guardian_artist,
        on="artist_key",
        how="left",
        validate="m:1"
    )
else:
    for col in [
        "article_count", "avg_article_words",
        "total_artist_mentions", "avg_text_length", "avg_tag_count"
    ]:
        merged[col] = np.nan

print("Итоговый размер:", merged.shape)

coverage = {}
for col in ["listeners", "mbid", "article_count"]:
    if col in merged.columns:
        coverage[col] = round(merged[col].notna().mean() * 100, 2)

display(pd.DataFrame(
    coverage.items(),
    columns=["external_feature", "coverage_%"]
))

display(merged.head())


## 14. Контроль качества после слияния

Проверим пропуски, дубликаты ключевых наблюдений и базовые статистики итогового набора. Пропуски во внешних API-признаках не удаляются: они означают, что для конкретного объекта внешний источник не дал совпадение. Это важно отличать от ошибки в основном наборе.

In [ ]:
quality = pd.DataFrame({
    "dtype": merged.dtypes.astype(str),
    "missing": merged.isna().sum(),
    "missing_%": (merged.isna().mean() * 100).round(2),
    "unique": merged.nunique(dropna=True)
}).sort_values("missing_%", ascending=False)

display(quality.head(30))

key_cols = [c for c in ["track_id", "week"] if c in merged.columns]
if key_cols:
    print(
        "Дубликатов по ключу",
        key_cols,
        ":",
        merged.duplicated(subset=key_cols).sum()
    )

summary_cols = [
    c for c in [
        "streams", "rank", "track_popularity", "artist_popularity",
        "artist_followers", "danceability", "energy", "valence",
        "tempo", "duration", "listeners", "playcount"
    ]
    if c in merged.columns
]

display(merged[summary_cols].describe().T)


## 15. Выбор структуры хранения

Для объединенного табличного набора выбран формат **Parquet**.

Почему он подходит:
1. сохраняет типы данных лучше CSV;
2. использует сжатое колоночное хранение;
3. удобен для последующего анализа отдельных признаков;
4. быстрее загружается при повторной работе с крупными таблицами;
5. поддерживается pandas и другими инструментами обработки больших данных.

TF-IDF является разреженной матрицей, поэтому она сохраняется отдельно в формате NPZ. Список признаков TF-IDF сохраняется в JSON.

Исходные данные из API не заменяют основной набор, а выступают слоями обогащения. Такая структура позволяет повторно обновлять отдельный источник без полной перестройки всех исходных данных.

In [ ]:
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_parquet_or_csv(df, stem):
    parquet_path = OUTPUT_DIR / f"{stem}.parquet"
    try:
        df.to_parquet(parquet_path, index=False)
        print("Сохранено:", parquet_path)
        return parquet_path
    except Exception as exc:
        csv_path = OUTPUT_DIR / f"{stem}.csv"
        df.to_csv(csv_path, index=False)
        print("Parquet недоступен:", exc)
        print("Резервно сохранено:", csv_path)
        return csv_path

save_parquet_or_csv(spotify, "spotify_clean")
save_parquet_or_csv(musicbrainz_df, "musicbrainz_artists")
save_parquet_or_csv(lastfm_df, "lastfm_tracks")
save_parquet_or_csv(guardian_df, "guardian_articles")
save_parquet_or_csv(merged, "music_merged")

if tfidf_matrix.shape[0] > 0:
    sparse.save_npz(OUTPUT_DIR / "guardian_tfidf.npz", tfidf_matrix)

    with open(
        OUTPUT_DIR / "guardian_tfidf_features.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(tfidf_feature_names, file, ensure_ascii=False, indent=2)


## 16. Скрипт обновления подготовленных данных

Для дальнейших этапов нужна возможность дополнять уже сохраненные данные новыми API-ответами. Используется функция upsert: новые строки добавляются, а записи с совпадающими ключами заменяются более свежей версией.

In [ ]:
def upsert_dataframe(existing, new_data, keys):
    """
    Объединяет старую и новую версии набора.
    При совпадении ключей сохраняется последняя (новая) запись.
    """
    if existing is None or existing.empty:
        return new_data.copy()

    if new_data is None or new_data.empty:
        return existing.copy()

    result = pd.concat(
        [existing, new_data],
        ignore_index=True,
        sort=False
    )

    return result.drop_duplicates(
        subset=keys,
        keep="last"
    ).reset_index(drop=True)

# Пример обновления Last.fm:
lastfm_updated = upsert_dataframe(
    lastfm_df,
    pd.DataFrame(columns=lastfm_df.columns),
    keys=["artist_key", "track_key"]
)

print("Размер Last.fm после тестового обновления:", lastfm_updated.shape)


## 17. Итог этапа 2.3

На этапе подготовки:

- загружен и исследован основной реляционный набор Spotify;
- проверены размер, типы данных, пропуски и дубликаты;
- описан физический смысл и аналитическая роль основных атрибутов;
- признаки распределены по восьми типам аналитических шкал;
- выполнена очистка Spotify и нормализация ключей;
- реализован разбор XML MusicBrainz;
- реализована загрузка и подготовка JSON Last.fm;
- реализованы сбор, очистка и аннотирование текстов The Guardian;
- выполнена TF-IDF-векторизация неструктурированных данных;
- определены общие атрибуты и агрегаты разных источников;
- реализовано слияние данных с контролем покрытия;
- выбран Parquet для хранения объединенных таблиц и NPZ для разреженной TF-IDF матрицы;
- подготовлена функция обновления данных по ключам.

Итоговый набор music_merged предназначен для следующего этапа 2.4, где будут применяться алгоритмы анализа данных и машинного обучения для проверки аналитической гипотезы.

**Важно:** для фактического заполнения блоков Last.fm и The Guardian перед финальным запуском необходимо задать соответствующие API-ключи и выполнить ноутбук сверху вниз. Секретные ключи в публичный репозиторий не сохраняются.